In [ ]:
from libs import *

In [ ]:
def load_tile_catalogue(csv_path: Path):
    """
    Load tile catalogue CSV into a list of (name, ra, dec) tuples.

    Parameters: 
        csv_path (Path): Path to csv file

    Returns: 
        List of tuples (name, ra, dec)
    
    """

    tiles = []

    with open(csv_path) as f:
        for row in csv.DictReader(f):
            tiles.append((row['tile'], float(row['ra']), float(row['dec'])))

    return tiles

def get_surrounding_tiles(tile_name, tiles):
    """
    Return the (up to) 8 surrounding tiles for a given CFIS tile.

    The CFIS grid is tilted: as Dec changes by +/-0.5°, the RA grid 
        shifts due to the cos(Dec) spacing. 3x3 neighbourhood is not 
        axis-aligned in RA/Dec and the top and bottom rows are offset
        in RA relative to the centre.

    Take:
      Same dec row  (ddec = 0.0): take the 2 nearest tiles in RA
      Adjacent rows (ddec = +/-0.5): take the 3 nearest tiles in RA

    Parameters:
        tile_name (str): Name of tile of interest.
        tiles (list) : list of tuples (name, ra, dec)

    Returns:
        (list of str): Names of the (upt to) 8 neighbours of the tile 
            of interest
    """

    target = next((t for t in tiles if t[0] == tile_name), None)
    if target is None:
        raise ValueError(f"Tile '{tile_name}' not found in catalogue.")
    _, ra0, dec0 = target

    # Bin candidates into dec rows
    by_dec = {}
    for name, ra, dec in tiles:
        if name == tile_name:
            continue
        ddec = round(dec - dec0, 4)
        if abs(ddec) not in (0.0, 0.5):
            continue
        by_dec.setdefault(ddec, []).append((name, ra, dec))

    neighbours = []
    for ddec_key, row_tiles in by_dec.items():
        # Sort by RA angular distance, wrap-safe
        row_tiles.sort(key=lambda t: abs((t[1] - ra0 + 180) % 360 - 180))
        n_keep = 2 if ddec_key == 0.0 else 3
        neighbours += [t[0] for t in row_tiles[:n_keep]]

    return neighbours

In [ ]:
tiles = load_tile_catalogue('cfis_r_tiles.csv')
print(get_surrounding_tiles('CFIS_LSB.017.242.r', tiles))